# Subject 4 · Predicting exam success: decision tree and logistic regression

**Course "Artificial Intelligence"** · Master 2 (ANG, LGC, DID) · Université Yahia Farès, Médéa

**Chapter 3** (supervised learning, evaluating a model) · **Mini-project, subject 4** (predictive modeling) · **Duration:** about 1 h 30

**What you will do**

- explore the fictional data of 200 undergraduate students of English: reading hours, attendance, homework handed in, books borrowed from the library, commuting time, and success (or not) in the final exam;
- train two models, a decision tree and a logistic regression, on some of the students, then evaluate them on the others (accuracy, confusion matrix);
- read the tree the computer learned, then discuss what it would be fair, or not, to do with such a model in a university.

**How to use this notebook.** Run each cell from top to bottom: click the ▶ button at its left, or press Shift+Enter. You never have to write code. Change only the fields of the cells titled "TO CHANGE". In Colab, the menu "Runtime → Run all" runs the whole notebook at once.

Before each code cell, a question asks you to **predict** the result: write down your answer, run the cell, then compare with the section "What you should see".

> **Fictional data.** The 200 students in this notebook are made up by a program, from a fixed seed and a rule that we show you. No real student is concerned: never quote these figures as a result.

The official syllabus speaks of "consumption data": here, library loans and reading time play that role. The same method can predict whether someone buys a book or attends a literary café, the example on the course site.

## Step 1 · Load the tools

This cell loads NumPy, pandas, matplotlib and the scikit-learn tools we need: splitting the data, the decision tree, logistic regression, standardization and the evaluation measures.

**Before you run it:** how many lines load tools from scikit-learn (the lines that start with `from sklearn`)?

In [ ]:
#@title Load the tools (libraries)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split             # split training and test
from sklearn.tree import DecisionTreeClassifier, plot_tree       # decision tree
from sklearn.linear_model import LogisticRegression              # logistic regression
from sklearn.preprocessing import StandardScaler                 # standardization
from sklearn.metrics import accuracy_score, precision_score, recall_score, ConfusionMatrixDisplay

COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]            # chart colors
CLASSES = ["fail", "pass"]                                       # 0 = fail, 1 = pass
print("Tools loaded.")

**What you should see:** `Tools loaded.` Five lines start with `from sklearn`.

**Question:** the last of these lines loads three measures: `accuracy_score`, `precision_score` and `recall_score`. What do they measure? Find their definitions in chapter 3 of the course.

## Step 2 · Generate 200 fictional students

Here is the **variable dictionary** of our data:

| Variable | Meaning | Values |
|---|---|---|
| `reading_hours` | hours of reading in English per week, outside class | 0 to 12 |
| `attendance` | share of the classes attended during the semester | in %, from 20 to 100 |
| `homework` | homework handed in, out of the 10 required | 0 to 10 |
| `library_loans` | books borrowed from the university library during the semester | number |
| `commute_min` | length of the journey between home and university | in minutes |
| `passed` | **the variable to predict**: at least 10/20 in the final exam | 1 = yes, 0 = no |

This cell randomly draws the first five variables. The commuting time is drawn first, because a student who lives far away misses classes more often: attendance goes down as the journey gets longer.

It also draws a sixth variable, `work_h`, the hours of paid work per week (small jobs, helping the family). In our fiction, students who live far away work more often. This variable will weigh on success, but the university does not know it: it **will not appear** in the table given to the models.

**Before you run it:** how many students out of 200 do you think live more than an hour away from the university in these data (the journey goes from 5 to 120 minutes)?

In [ ]:
#@title TO CHANGE: the random seed, then run (the explanatory variables)
seed = 2026 #@param {type:"integer"}
rng = np.random.default_rng(seed)        # reproducible random generator
n = 200                                  # number of fictional students
commute_min = rng.integers(5, 121, n)                                           # 5 to 120 minutes
attendance = np.clip(rng.normal(100 - 0.3 * commute_min, 12), 20, 100).round()  # goes down with the commute
reading_hours = np.clip(rng.normal(4, 2.5, n), 0, 12).round()
homework = rng.binomial(10, 0.6, n)                                             # out of 10
library_loans = rng.poisson(0.5 + 0.4 * reading_hours)                          # readers borrow more
work_h = rng.poisson(commute_min / 12)          # hours of paid work: UNKNOWN to the university
print("Students more than an hour away:", (commute_min > 60).sum())
print("Average attendance (%):", attendance.mean().round(1))
print("Average paid work (hours per week):", work_h.mean().round(1))

**What you should see:** `Students more than an hour away: 102`, `Average attendance (%): 80.9` and `Average paid work (hours per week): 5.0`. About half of the fictional students live far away, as in a region where many come from distant towns.

**Question:** why is it realistic for attendance to depend on the commute? Name another variable of students' lives that could play the same role.

We still have to generate **success**. The next cell applies a hidden rule: a score that goes up with reading, attendance, homework and loans, and goes down with paid work, then the logistic function `1 / (1 + exp(−score))`, which turns this score into a probability between 0 and 1. Finally, chance decides: a student with a 70% chance of passing still fails three times out of ten. This share of chance will prevent any model from being perfect.

**Before you run it:** read the line `score = …`. Which variable of the dictionary does **not** appear in it? And which variable of the rule will **not** be in the `students` table? Remember these two variables: we will meet them again in the ethics discussion.

In [ ]:
#@title Generate exam success (hidden rule + chance), then the table
score = -16 + 1.2 * reading_hours + 0.13 * attendance + 0.8 * homework + 0.1 * library_loans - 0.4 * work_h
probability = 1 / (1 + np.exp(-score))                   # logistic function: between 0 and 1
passed = (rng.random(n) < probability).astype(int)       # 1 = pass, 0 = fail
students = pd.DataFrame({           # the table known to the university: without work_h
    "reading_hours": reading_hours.astype(int), "attendance": attendance.astype(int),
    "homework": homework, "library_loans": library_loans, "commute_min": commute_min, "passed": passed})
variables = ["reading_hours", "attendance", "homework", "library_loans", "commute_min"]
print("Pass rate:", students["passed"].mean().round(2))
students.head(6)

**What you should see:** `Pass rate: 0.7` (140 students out of 200), then the first 6 rows. The first student reads 2 hours, attends 54% of the classes, lives 103 minutes away: they failed (`0`). The fourth one (row 3) passed with 72% attendance and a 79-minute commute: the chance in the rule is at work.

The variable missing from the rule is **`commute_min`**: in our fiction, the commute affects success only **indirectly**, because it lowers attendance and goes together with paid work. And the variable of the rule missing from the table is **`work_h`**.

**Question:** the fourth student passed although their situation looks like that of many failing students. Would a model that got this student wrong be a "bad" model?

## Step 3 · Explore: those who pass and those who fail

Before modeling, we compare the two groups: the mean of each variable, then a **box plot** for each variable. The box contains the middle half of the students, the line in the middle is the median, the "whiskers" show the range of usual values, isolated circles are rare values.

**Before you run it:** for which variable will the gap between the two groups be the most visible? Do the students who fail live further away, on average?

In [ ]:
#@title Compare the students who fail (0) and who pass (1)
print(students.groupby("passed")[variables].mean().round(1))
fig, axes = plt.subplots(1, 5, figsize=(14, 3.5))
for ax, v in zip(axes, variables):
    fails = students.loc[students["passed"] == 0, v]
    passes = students.loc[students["passed"] == 1, v]
    ax.boxplot([fails, passes])
    ax.set_xticks([1, 2], CLASSES)
    ax.set_title(v)
plt.suptitle("Each box sums up a group of students (fictional data)")
plt.tight_layout()
plt.show()

**What you should see:** a table of means and five pairs of boxes. Failing students: 2.2 hours of reading, 70.8% attendance, 5.3 pieces of homework, 1.3 loans, 78.5 minutes of commute. Passing students: 4.8 hours, 85.3%, 5.9 pieces of homework, 2.6 loans, 55.8 minutes. The clearest gaps concern reading and attendance; homework differs little. And the failing students live **23 minutes further away** on average, although the commute is not in the rule.

**Question:** how do you explain this commuting gap between the two groups? Find two reasons by rereading step 2.

## Step 4 · Split training and test

A model is never evaluated on the examples used to train it, just as a student is not judged on questions whose answers they know by heart. So we keep some of the students aside: the **test set**. `stratify=y` keeps the same pass rate in both parts.

We also compute the accuracy of a **naive model**, which always predicts the most frequent answer without looking at anything: a good model must do better.

**Before you run it:** with 25% of the 200 students kept for the test, how many students will be used for training? What accuracy will the naive model have?

In [ ]:
#@title TO CHANGE: the share kept for the test, then run (split)
test_share = 0.25 #@param {type:"slider", min:0.1, max:0.5, step:0.05}
X = students[variables]           # what the model looks at
y = students["passed"]            # what it must predict
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_share, random_state=0, stratify=y)
print("Students for training:", len(X_train))
print("Students for testing: ", len(X_test))
naive = y_test.value_counts(normalize=True).max()  # share of the most frequent answer
print("Accuracy of the naive model:", round(naive, 2))

**What you should see:** 150 students for training, 50 for testing (15 fails and 35 passes), and an accuracy of **0.7** for the naive model: by always answering "pass", it is right for 35 test students out of 50.

**Question:** is a model with 72% accuracy on these data a good model? Compare it with the naive model.

## Step 5 · The decision tree

A decision tree learns a series of questions of the form "variable ≤ threshold?". It is an IF … THEN rule base, like an expert system, but learned from the examples. The **depth** is the maximum number of questions asked in a row.

**Before you run it:** will the tree be more accurate on the training students (which it has seen) or on the test students (which it has never seen)?

In [ ]:
#@title TO CHANGE: the depth of the tree, then run (decision tree)
depth = 3 #@param {type:"slider", min:1, max:8, step:1}
tree = DecisionTreeClassifier(max_depth=depth, random_state=0)
tree.fit(X_train, y_train)                      # training
tree_pred = tree.predict(X_test)                # predictions for students never seen
print("Accuracy on training:", round(accuracy_score(y_train, tree.predict(X_train)), 2))
print("Accuracy on test:    ", round(accuracy_score(y_test, tree_pred), 2))

**What you should see:** 0.87 on training, 0.78 on test. The tree does better than the naive model (0.7), but it is stronger on the examples it knows than on new ones.

**Question:** what would an accuracy of 1.0 on training and 0.6 on test mean?

This cell **draws the tree**. Each box asks a question; you go down to the left if the answer is "true", to the right if it is "false". In each box: the question, `samples` (the number of training students who get there), `value` (how many fails and passes among them) and `class` (the majority answer). The color leans towards orange for fail, towards blue for pass.

**Before you run it:** which variable will the tree choose for its very first question?

In [ ]:
#@title Draw the decision tree
fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(tree, feature_names=variables, class_names=CLASSES, filled=True, rounded=True,
          impurity=False, fontsize=9, ax=ax)
ax.set_title("The learned tree: left if true, right if false")
plt.show()

**What you should see:** the first question is `reading_hours <= 1.5` (150 students: 45 fails, 105 passes).

- On the left, those who read one hour at most (25 students). The tree then asks a question about the **commute**: the 18 who live more than 59.5 minutes away all failed, hence the rule IF reading_hours ≤ 1.5 AND commute_min > 59.5 THEN fail. Those who live closer are classified "pass" if their attendance is above 75.5%.
- On the right, those who read at least 2 hours (125 students). IF attendance is at most 69% AND reading at most 5 hours, THEN fail; otherwise, pass.

The two leaves on the right give the same answer ("pass"): the tree made the groups more homogeneous without changing its decision.

**Question:** what do you think of the rule that uses `commute_min`? Write down in English another IF … THEN rule read from the tree: does it seem plausible from a teaching point of view?

## Step 6 · Overfitting

The deeper the tree, the more questions it asks, and the better it fits the training students, until it learns their accidents by heart. This is **overfitting**. This cell trains a tree for each depth from 1 to 10 and compares the two accuracies.

**Before you run it:** how will the "training" curve change as the depth increases? And the "test" curve?

In [ ]:
#@title Compare accuracy for each depth of the tree (1 to 10)
rows = []
for d in range(1, 11):
    attempt = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    rows.append({"depth": d, "training": accuracy_score(y_train, attempt.predict(X_train)),
                 "test": accuracy_score(y_test, attempt.predict(X_test))})
curve = pd.DataFrame(rows).round(2)
print(curve.to_string(index=False))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(curve["depth"], curve["training"], marker="o", color=COLORS[0], label="training")
ax.plot(curve["depth"], curve["test"], marker="s", color=COLORS[1], label="test")
ax.axhline(naive, color="gray", linestyle="--", linewidth=1, label="naive model")
ax.set(xlabel="Depth of the tree", ylabel="Accuracy")
ax.legend()
plt.show()

**What you should see:** the training accuracy keeps rising, from 0.8 (depth 1) to 1.0 (depths 9 and 10): in the end, the tree knows each student by heart. The test accuracy reaches 0.8 at best (depth 4), drops to 0.66 (depth 5), then stays between 0.74 and 0.78. The growing gap between the two curves is the signature of overfitting.

**Question:** which depth would you choose? Why is a shallower tree also easier to explain to a student or a colleague?

## Step 7 · Logistic regression

Despite its name, logistic regression is used to **classify**. It looks for a weight (a **coefficient**) for each variable, adds everything up into a score, then turns this score into a probability with the same logistic function as in step 2. We first standardize the variables (as in notebook 02) so that their coefficients can be compared. The scaler learns the mean and standard deviation from the training students only.

**Before you run it:** will logistic regression do better or worse than the tree on the test? Remember: the hidden rule is itself a logistic formula.

In [ ]:
#@title Train and evaluate a logistic regression
scaler = StandardScaler().fit(X_train)           # means and standard deviations of training
X_train_std = scaler.transform(X_train)          # standardized variables
X_test_std = scaler.transform(X_test)
logreg = LogisticRegression()
logreg.fit(X_train_std, y_train)                 # training
logreg_pred = logreg.predict(X_test_std)
print("Accuracy on training:", round(accuracy_score(y_train, logreg.predict(X_train_std)), 2))
print("Accuracy on test:    ", round(accuracy_score(y_test, logreg_pred), 2))

**What you should see:** `Accuracy on training: 0.92` and `Accuracy on test: 0.84`. Logistic regression does better than the tree on the test (0.84 against 0.78) and much better than the naive model (0.7).

**Question:** with 50 test students, one student is worth 2 points of accuracy. How many students does the gap between the two models correspond to? Is it a big difference?

The **coefficients** tell in which direction each variable pushes the prediction: positive, it makes passing more likely; negative, less likely. Since the variables are standardized, we can compare their sizes. Be careful: a coefficient shows an **association**, not a cause.

**Before you run it:** which coefficient will be the largest? Will the coefficient of `commute_min` be zero, since the commute is not in the hidden rule?

In [ ]:
#@title Read and draw the coefficients of the logistic regression
coefficients = pd.Series(logreg.coef_[0], index=variables).sort_values()
print(coefficients.round(2).to_string())
bar_colors = [COLORS[1] if c < 0 else COLORS[0] for c in coefficients]     # orange if negative
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(coefficients.index, coefficients.values, color=bar_colors)
ax.axvline(0, color="gray", linewidth=1)
ax.set_xlabel("Coefficient (standardized variables)")
ax.set_title("Direction and strength of the association with passing")
plt.show()

**What you should see:** `commute_min` −0.45 (the only negative coefficient, orange bar), `library_loans` 0.33, `homework` 1.23, `attendance` 1.51, `reading_hours` 2.02 (the strongest). The model has correctly found that reading matters most. But it gives the commute a **negative** weight, although the commute is not in the hidden rule.

**Question:** so a student who lives far away will be judged less likely to pass, **with the same attendance**. Which hidden variable from step 2 explains this weight? Is it fair to a student who lives far away but does not work?

## Step 8 · Which errors? The confusion matrices

Accuracy does not say **which** errors the model makes. The confusion matrix crosses the true answer (rows) with the prediction (columns). The two cells on the diagonal are the correct answers; the other two are the errors: a student who passed but whom the model expected to fail, or a failing student whom the model did not see coming.

**Before you run it:** the 50 test students include 15 fails and 35 passes. Which type of error will be the most frequent?

In [ ]:
#@title Draw the confusion matrices of the two models (test set)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name, pred in zip(axes, ["Decision tree", "Logistic regression"], [tree_pred, logreg_pred]):
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=CLASSES,
                                            cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(f"{name}: accuracy {accuracy_score(y_test, pred):.2f}")
    ax.set_xlabel("Model prediction")
    ax.set_ylabel("True answer")
plt.tight_layout()
plt.show()

**What you should see:**

- Decision tree: 9 fails correctly predicted, **6 fails predicted as "pass"**, 5 passes predicted as "fail", 30 passes correctly predicted (9 + 30 = 39 correct answers out of 50, that is 0.78).
- Logistic regression: 13 fails correctly predicted, **2 fails predicted as "pass"**, 6 passes predicted as "fail", 29 passes correctly predicted (42 out of 50, that is 0.84).

The two models do not make the same errors: the tree misses 6 fails out of 15; logistic regression misses only 2, but it wrongly predicts failure for 6 students who passed.

**Question:** for a student, which error is worse: being predicted to fail when they would have passed, or not being spotted when they are going to fail? Does the answer depend on what the university does with the prediction?

If the university wants to **spot students in difficulty**, the category we are looking for (the "positive" one, like spam in the course) is failure. Two measures from the course then answer two questions:

- **recall**: what share of the students who really fail does the model spot?
- **precision**: when the model predicts a failure, is it right?

**Before you run it:** according to the matrices, which model will have the better recall for failure?

In [ ]:
#@title Recall and precision for the "fail" category (0)
for name, pred in [("Decision tree", tree_pred), ("Logistic regression", logreg_pred)]:
    recall = recall_score(y_test, pred, pos_label=0)
    precision = precision_score(y_test, pred, pos_label=0)
    print(name)
    print(f"   spots {recall:.0%} of the failing students (recall)")
    print(f"   is right in {precision:.0%} of the cases when it predicts a failure (precision)")

**What you should see:** the tree spots 60% of the failing students (9 out of 15) and is right in 64% of the cases when it predicts a failure (9 out of 14). Logistic regression spots 87% of the failures (13 out of 15), with a precision of 68% (13 out of 19): it finds almost all the students in difficulty, but nearly one time in three, the student it flags would have passed.

**Question:** imagine we set the model to predict even more failures. Recall would go up. What would happen to precision, and what would that mean for the students?

## Step 9 · Predict for a new student

Set the sliders to describe a fictional student, then run the cell: logistic regression gives a probability of passing, the tree gives an answer.

**Before you run it:** with the default values (2 hours of reading, 75% attendance, 6 pieces of homework, 1 loan, 60 minutes of commute), does this student have more or less than a one-in-two chance of passing, in your opinion?

In [ ]:
#@title TO CHANGE: describe a new student, then run (prediction)
new_reading_hours = 2 #@param {type:"slider", min:0, max:12, step:1}
new_attendance = 75 #@param {type:"slider", min:20, max:100, step:5}
new_homework = 6 #@param {type:"slider", min:0, max:10, step:1}
new_library_loans = 1 #@param {type:"slider", min:0, max:10, step:1}
new_commute = 60 #@param {type:"slider", min:5, max:120, step:5}
new_student = pd.DataFrame([[new_reading_hours, new_attendance, new_homework,
                             new_library_loans, new_commute]], columns=variables)
proba = logreg.predict_proba(scaler.transform(new_student))[0, 1]
print(f"Logistic regression: probability of passing = {proba:.2f}")
print("Decision tree:      ", CLASSES[tree.predict(new_student)[0]])

**What you should see:** `Logistic regression: probability of passing = 0.38` and `Decision tree: pass`. The two models disagree! Follow this student in the tree: they read more than 1.5 hours, attend more than 69% of the classes and read 2.5 hours at most; they end up in the leaf `samples = 20, value = [7, 13]`, where 13 students out of 20 passed.

**Question:** which of the two models would you believe for this student? Raise only the reading hours to 4 and run the cell again: what happens to the probability and to the tree's answer?

## Step 10 · Ethics discussion: same student, two addresses

Let us take the student of step 9 again and change **a single** piece of information: where they live. The first one lives 10 minutes from the university, the second one 110 minutes away (a distant town in the region). Everything else is identical: same reading, **same attendance**, same homework.

**Before you run it:** will the two probabilities be equal? If they differ, by how much?

In [ ]:
#@title Same student, two places of residence
for minutes in [10, 110]:
    student = new_student.copy()
    student["commute_min"] = minutes             # we only change the commute
    p = logreg.predict_proba(scaler.transform(student))[0, 1]
    print(f"Lives {minutes:3d} minutes away: probability of passing = {p:.2f}")

**What you should see:** 0.55 for the student who lives 10 minutes away, 0.23 for the one who lives 110 minutes away. With identical behavior, the model takes 32 points of probability away from the student who lives far away: the first would be predicted "pass", the second "fail".

Why? In our fiction, the real cause is **paid work**, which the model does not see. So it falls back on what goes with it: the commute. The commute becomes a **"proxy" variable** (a stand-in indicator). A student who lives far away but does not work is thus penalized for a cause that is not theirs.

**Question:** if the university kept tutoring for students whose probability is below 0.3, which of the two would get it? And if it kept an excellence grant for probabilities above 0.5?

### Predicting is not judging: some landmarks

- **"Proxy" variables.** Commuting time is not a skill: it tells where someone lives, and so often where they come from. When the real cause (here, paid work) is not measured, the model relies on what goes with it, and can therefore penalize a geographical or social origin without anyone having decided it. Removing the column is not always enough: attendance, too, carries the trace of the commute. Pierre Bourdieu (*Distinction*, 1979) showed that cultural practices are linked to educational capital and social background: a model learns these inequalities and may reproduce them.
- **A real precedent: the Ofqual algorithm (England, 2020).** Since the *A-level* exams were cancelled because of Covid-19, grades were computed by an algorithm that adjusted the grades proposed by teachers according to each school's past results. Nearly 40% of the grades were lowered, above all in schools with historically weak results. After student protests, the government abandoned these grades on 17 August 2020 and went back to the teachers' grades.
- **The Pygmalion effect.** Robert Rosenthal and Lenore Jacobson (*Pygmalion in the Classroom*, 1968) argued, in a famous and debated study, that teachers' expectations influence students' progress. An "at risk" label displayed by software can become a self-fulfilling prophecy.
- **Errors that affect people.** The confusion matrices showed it: even the better model is wrong about 8 test students out of 50. Behind each cell, there is a student who was misdirected, or forgotten.
- **The law.** In Algeria, law no. 18-07 of 10 June 2018 regulates the processing of personal data. In the European Union, the AI Act (2024) classifies as "high-risk" the AI systems used in education to decide on admission or to evaluate learners' outcomes: they must be supervised by humans, documented and based on quality data.
- **Predicting is not explaining.** A coefficient shows an association, not a cause: borrowing library books does not make students pass, it goes together with reading.

## Your turn

Change only the fields, then run the cells again from the one you changed (in Colab: "Runtime → Run after").

1. **Step 5:** set the depth to 1, then to 8. Also rerun the drawing of the tree. Which one would you explain to a student? Which one is the most accurate on the test?
2. **Step 4:** set the test share to 0.4. Do the accuracies change a lot? What do you conclude about the reliability of a result obtained with few students?
3. **Step 9:** find the number of reading hours from which the probability of passing goes above 0.5, then repeat the experiment with a commute of 5 minutes and of 120 minutes. Does the threshold change?

## For the mini-project: your own data

Subject 4 of the mini-project suggests, for example, a questionnaire that ends with "Did you attend a literary event this year? (yes/no)". To use your answers:

- prepare a CSV file with **one row per respondent**, numeric columns (age, books read per year, distance, etc.) and one column **coded 1 or 0**: the variable to predict;
- an anonymous questionnaire, filled in with the respondents' consent; with only a few dozen answers, be very careful in your conclusions;
- write the name of the column to predict in the field, tick the box and run the cell: in Colab, a window asks you for the file; in Jupyter, put it next to the notebook under the name `my_data.csv`;
- then run steps 4 to 8 again (the names "fail" and "pass" will simply stand for 0 and 1). Steps 9 and 10 only work with the fictional data.

**Before you run it:** if you leave the box unticked, what will the cell do?

In [ ]:
#@title TO CHANGE: load your own data (CSV file)
load_my_file = False #@param {type:"boolean"}
column_to_predict = "attended" #@param {type:"string"}
if load_my_file:
    try:
        from google.colab import files       # only exists in Colab
        upload = files.upload()              # window to choose the file
        file_name = list(upload.keys())[0]
    except ImportError:                      # Jupyter: file placed next to the notebook
        file_name = "my_data.csv"
    students = pd.read_csv(file_name).dropna().rename(columns={column_to_predict: "passed"})
    variables = [c for c in students.select_dtypes("number").columns if c != "passed"]
    print(len(students), "complete answers; explanatory variables:", variables)
else:
    print("Box not ticked: the notebook keeps the fictional data.")

**What you should see:** with the box unticked, `Box not ticked: the notebook keeps the fictional data.` With your file, the number of complete answers and the list of your explanatory variables.

**Question:** among the questions of your own questionnaire, which one could act as a "proxy" variable, like the commute here?

## To hand in

Answer in writing, in English, in a few lines for each question.

1. **Compare.** Compare the decision tree, logistic regression and the naive model: accuracy, confusion matrices, recall and precision for failure. Which one would you choose, and for what use?
2. **Read the tree.** Write down in sentences two IF … THEN rules read from the depth-3 tree. Are they plausible for a language teacher? Which one surprises you?
3. **Ethical opinion.** Your university wants to use such a model to spot "at risk" students as early as November. Write a reasoned opinion of about ten lines: possible errors, the commuting variable, the Pygmalion effect, what should be required (human oversight, transparency, consent, the right to challenge).

## Key points

- In **supervised learning**, the model learns from labeled examples (training set) and is evaluated on examples it has never seen (test set), in comparison with a **naive model**.
- The **decision tree** learns readable IF … THEN rules, but **overfits** if it is too deep; **logistic regression** gives a probability and coefficients, which show an association and not a cause.
- Accuracy is not enough: the **confusion matrix** shows which errors the model makes. These errors affect people: "proxy" variables, labels, data protection law.

**Lessons on the course site:** [Mini-project, subject 4: predictive modeling](https://progremer04.github.io/canva-au-for-ang-/en/#mp-sujet-4) · [Chapter 3: evaluating a model, overfitting](https://progremer04.github.io/canva-au-for-ang-/en/#c3-ml) · [Chapter 3: classification by features](https://progremer04.github.io/canva-au-for-ang-/en/#c3-algorithmes)